In [9]:
# Imports
import pdfplumber
import fitz  # PyMuPDF
from pathlib import Path
import time
import json
from datetime import datetime, date, time as dt_time
import re
from collections import defaultdict
from pprint import pprint

# Configuration
PDF_DIR = Path('../feuilles_match')
PDF_FILES = list(PDF_DIR.glob('*.pdf'))
print(f"PDFs trouvés: {len(PDF_FILES)}")
for f in PDF_FILES:
    print(f"  - {f.name}")

PDFs trouvés: 9
  - LIIDF_PMAB001.pdf
  - LIIDF_PFAA002.pdf
  - LIIDF_PMAA005.pdf
  - LIIDF_PMAA003.pdf
  - LIIDF_PFAA001.pdf
  - LIIDF_PMAA001.pdf
  - LIIDF_PMAA004.pdf
  - LIIDF_PMAA002.pdf
  - LIIDF_1MAA001.pdf


## 1. Exploration du PDF avec pdfplumber

In [10]:
# Ouvrir un PDF de test
pdf_path = PDF_FILES[0] if PDF_FILES else None
print(f"Analyse de: {pdf_path}")

with pdfplumber.open(pdf_path) as pdf:
    page = pdf.pages[0]
    
    # Dimensions de la page
    print(f"\nDimensions: {page.width} x {page.height}")
    
    # Extraire tout le texte
    text = page.extract_text()
    print(f"\n=== Texte extrait (premiers 2000 caractères) ===\n")
    print(text[:2000] if text else "Aucun texte")

Analyse de: ../feuilles_match/LIIDF_PMAB001.pdf

Dimensions: 841.89 x 595.28



=== Texte extrait (premiers 2000 caractères) ===

PMA - CHAMPIONNAT PRE-NATIONAL SENIOR MASCULIN : POULE A Match: PMAB001 - Jour:
Ville: Mardi à h
Salle: SENIOR | MASCULIN
Ligue ILE-DE-FRANCE xxxxx xxxxx
S Début: Fin: S Début: Fin:
Ordre de Service E I II III IV V VI I II III IV V VI E I II III IV V VI I II III IV V VI
Formation de Départ
Joueur N° T T
Remplaçants
Score
1 5 1 2
2 6 T T T T
Tours au service
3 7
4 8
S Début: Fin: S Début: Fin:
Ordre de Service E I II III IV V VI I II III IV V VI E I II III IV V VI I II III IV V VI
Formation de Départ
Joueur N° T T
Remplaçants
Score
1 5 3 4
2 6 T T T T
Tours au service
3 7
4 8
S xxxxx xxxxx
E I II III IV V VI I II III IV V VI I II III IV V VI N° Nom Prénom Licence N° Nom Prénom Licence
T
1
T T T
DEMANDE NON FONDEE REMARQUES
SANCTIONS
EQU.A EQU.B
A P E D A/B Set Score
LIBEROS
APPROBATION RESULTATS
Arbitres NOM Prénom Ligue Licence Signature Equipe A Equipe B OFFICIELS
1er T R G P Durée par Set P G R T
0 0 0 0 0' 0 0 0 0
2ème Début Fin Dur

In [11]:
# Analyser les caractères avec leurs positions
with pdfplumber.open(pdf_path) as pdf:
    page = pdf.pages[0]
    chars = page.chars[:100]  # Premiers 100 caractères
    
    print("=== Échantillon de caractères avec positions ===")
    for c in chars[:20]:
        print(f"'{c['text']}' @ ({c['x0']:.1f}, {c['top']:.1f}) size={c.get('size', 0):.1f}")

=== Échantillon de caractères avec positions ===
'L' @ (14.2, 59.4) size=6.0
'i' @ (17.8, 59.4) size=6.0
'g' @ (19.5, 59.4) size=6.0
'u' @ (23.2, 59.4) size=6.0
'e' @ (26.8, 59.4) size=6.0
' ' @ (30.2, 59.4) size=6.0
'I' @ (31.8, 59.4) size=6.0
'L' @ (33.5, 59.4) size=6.0
'E' @ (37.2, 59.4) size=6.0
'-' @ (41.2, 59.4) size=6.0
'D' @ (43.2, 59.4) size=6.0
'E' @ (47.5, 59.4) size=6.0
'-' @ (51.5, 59.4) size=6.0
'F' @ (53.5, 59.4) size=6.0
'R' @ (57.2, 59.4) size=6.0
'A' @ (61.5, 59.4) size=6.0
'N' @ (65.8, 59.4) size=6.0
'C' @ (70.2, 59.4) size=6.0
'E' @ (74.5, 59.4) size=6.0
'P' @ (116.2, 29.1) size=10.0


In [12]:
# Extraction des tables
with pdfplumber.open(pdf_path) as pdf:
    page = pdf.pages[0]
    tables = page.extract_tables()
    
    print(f"\n=== {len(tables)} tables détectées ===")
    for i, table in enumerate(tables):
        print(f"\n--- Table {i+1} ({len(table)} lignes) ---")
        for row in table[:10]:  # Premières 10 lignes
            print(row)


=== 5 tables détectées ===

--- Table 1 (43 lignes) ---
[None, None, None, None, None, None, None, None, None, None, None, 'S\nE\nT\n1', None, 'Début:', None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, 'Fin:', None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None]
['Ordre de Service', None, None, None, None, None, None, None, None, None, None, None, None, 'I', None, 'II', None, None, 'III', None, 'IV', None, 'V', None, 'VI', None, '', None, None, None, None, 'I', None, 'II', None, 'III', None, 'IV', None, 'V', None, 'VI', None, '', None, None, None, None, None]
['Formation de Départ', None, None, None, None, None, None, None, None, None, None, None, None, '', None, '', None, None, '', None, '', None, '', None, '', None, None, None, None, None, None, '', None, '', None, '', None, '', None, '', None, '', None, None, None, None, None, None, None]
['Remplaçants', None, None, None, None, Non

In [13]:
# Identifier les zones du document par regroupement de mots
with pdfplumber.open(pdf_path) as pdf:
    page = pdf.pages[0]
    words = page.extract_words()
    
    # Grouper par zones verticales
    zones = defaultdict(list)
    for w in words:
        y_zone = int(w['top'] // 50) * 50  # Grouper par tranches de 50
        zones[y_zone].append(w)
    
    print("=== Zones verticales ===")
    for y in sorted(zones.keys()):
        texts = [w['text'] for w in zones[y][:10]]
        print(f"Y={y:3d}: {' | '.join(texts[:8])}..." if len(texts) > 8 else f"Y={y:3d}: {' | '.join(texts)}")

=== Zones verticales ===
Y=  0: PMA | - | CHAMPIONNAT | PRE-NATIONAL | SENIOR | MASCULIN | : | POULE...
Y= 50: Ligue | ILE-DE-FRANCE | xxxxx | xxxxx | S | Début: | Fin: | S...
Y=100: Joueur | N° | T | T | Remplaçants | Score | 1 | 5...
Y=150: 4 | 8 | S | Début: | Fin: | S | Début: | Fin:...
Y=200: Remplaçants | Score | 1 | 5 | 3 | 4 | 2 | 6...
Y=250: 4 | 8 | S | xxxxx | xxxxx | E | I | II...
Y=300: 1 | T | T | T
Y=350: DEMANDE | NON | FONDEE | REMARQUES | SANCTIONS | EQU.A | EQU.B | A...
Y=400: LIBEROS | APPROBATION | RESULTATS | Arbitres | NOM | Prénom | Ligue | Licence...
Y=450: 0 | 0 | 0 | 0 | 0' | 0 | 0 | 0...
Y=500: Lignes | Capitaines | Entraineur | Entraineur


## 2. Développement du Parseur pdfplumber (V3)

In [14]:
class MatchSheetParserV3:
    """
    Parser V3 pour les feuilles de match FFVB.
    Utilise pdfplumber pour l'extraction de texte et tables.
    """
    
    def __init__(self):
        self.name = "MatchSheetParserV3"
        self.version = "3.0.0"
    
    def parse(self, pdf_path: Path) -> dict:
        """Parse un PDF de feuille de match."""
        start_time = time.time()
        result = {
            'success': False,
            'data': {},
            'errors': [],
            'warnings': [],
            'parse_time_ms': 0
        }
        
        try:
            with pdfplumber.open(pdf_path) as pdf:
                page = pdf.pages[0]
                
                # Extraire les mots avec positions
                words = page.extract_words()
                
                # Extraire les tables
                tables = page.extract_tables()
                
                # Parser les différentes sections
                data = {}
                data['header'] = self._parse_header(words)
                data['equipes'] = self._parse_equipes(words, tables)
                data['resultat'] = self._parse_resultat(words)
                data['sets'] = self._parse_sets(words, tables)
                data['arbitres'] = self._parse_arbitres(words)
                data['joueurs'] = self._parse_joueurs(words, tables)
                
                result['data'] = data
                result['success'] = True
                
        except Exception as e:
            result['errors'].append(str(e))
        
        result['parse_time_ms'] = (time.time() - start_time) * 1000
        return result
    
    def _parse_header(self, words: list) -> dict:
        """Parse l'en-tête du match."""
        header = {
            'ligue': None,
            'competition': None,
            'code_match': None,
            'journee': None,
            'date': None,
            'heure': None,
            'lieu': None,
            'salle': None,
            'saison': None,
        }
        
        # Filtrer les mots du header (y < 100)
        header_words = [w for w in words if w['top'] < 100]
        text_line = ' '.join(w['text'] for w in header_words)
        
        # Ligue
        ligue_match = re.search(r'(Ligue[^\n]+)', text_line)
        if ligue_match:
            header['ligue'] = ligue_match.group(1).strip()
        
        # Compétition et code match
        for w in header_words:
            if 'CHAMPIONNAT' in w['text'].upper():
                header['competition'] = w['text']
            if re.match(r'Match:', w['text']):
                match = re.search(r'Match:\s*(\w+)', w['text'])
                if match:
                    header['code_match'] = match.group(1)
        
        # Date - chercher pattern jour de semaine + date
        date_pattern = r'(Lundi|Mardi|Mercredi|Jeudi|Vendredi|Samedi|Dimanche)\s+\d+\s+\w+\s+\d{4}'
        date_match = re.search(date_pattern, text_line)
        if date_match:
            header['date'] = date_match.group(0)
        
        # Heure
        heure_match = re.search(r'(\d{1,2}h\d{2})', text_line)
        if heure_match:
            header['heure'] = heure_match.group(1)
        
        # Ville et Salle
        ville_match = re.search(r'Ville:\s*([^\n]+)', text_line)
        if ville_match:
            header['lieu'] = ville_match.group(1).strip()
        
        salle_match = re.search(r'Salle:\s*([^\n]+)', text_line)  
        if salle_match:
            header['salle'] = salle_match.group(1).strip()
        
        # Saison
        saison_match = re.search(r'(\d{4}-\d{4})', text_line)
        if saison_match:
            header['saison'] = saison_match.group(1)
        
        return header
    
    def _parse_equipes(self, words: list, tables: list) -> dict:
        """Parse les noms des équipes."""
        equipes = {'equipe_a': None, 'equipe_b': None}
        
        # Chercher les gros textes (noms d'équipes) vers y=60-80
        team_words = [w for w in words if 55 < w['top'] < 85 and w.get('height', 0) > 10]
        team_words.sort(key=lambda w: w['x0'])
        
        if team_words:
            # Séparer gauche/droite
            mid_x = max(w['x1'] for w in words) / 2
            left = [w for w in team_words if w['x0'] < mid_x]
            right = [w for w in team_words if w['x0'] >= mid_x]
            
            if left:
                equipes['equipe_a'] = ' '.join(w['text'] for w in left)
            if right:
                equipes['equipe_b'] = ' '.join(w['text'] for w in right)
        
        return equipes
    
    def _parse_resultat(self, words: list) -> dict:
        """Parse le résultat du match."""
        resultat = {
            'vainqueur': None,
            'score_final': None,
            'duree_totale': None
        }
        
        # Chercher dans la zone basse (y > 500)
        result_words = [w for w in words if w['top'] > 500]
        result_text = ' '.join(w['text'] for w in result_words)
        
        # Score final (format X/Y)
        score_match = re.search(r'(\d)/(\d)', result_text)
        if score_match:
            resultat['score_final'] = f"{score_match.group(1)}/{score_match.group(2)}"
        
        # Vainqueur - chercher après "Vainqueur"
        vainqueur_idx = None
        for i, w in enumerate(result_words):
            if 'Vainqueur' in w['text']:
                vainqueur_idx = i
                break
        
        if vainqueur_idx is not None:
            # Chercher le nom après
            for w in result_words[vainqueur_idx+1:vainqueur_idx+5]:
                if len(w['text']) > 3 and '/' not in w['text'] and not w['text'].isdigit():
                    resultat['vainqueur'] = w['text']
                    break
        
        # Durée
        duree_match = re.search(r'(\d+h\d+)', result_text)
        if duree_match:
            resultat['duree_totale'] = duree_match.group(1)
        
        return resultat
    
    def _parse_sets(self, words: list, tables: list) -> list:
        """Parse les scores des sets."""
        sets = []
        
        # Chercher dans les tables la grille des scores
        for table in tables:
            # Chercher une table avec des headers comme "Set", "1", "2", etc.
            for row in table:
                if row and any(cell and 'Set' in str(cell) for cell in row):
                    # C'est potentiellement la table des scores
                    continue
        
        # Méthode alternative: chercher par position
        # Zone grille typique: y entre 445 et 530, x entre 420 et 560
        grid_words = [w for w in words if 440 < w['top'] < 540 and 420 < w['x0'] < 560]
        
        # Grouper par ligne
        lines = defaultdict(list)
        for w in grid_words:
            y_key = round(w['top'] / 5) * 5
            lines[y_key].append(w)
        
        # Parser chaque ligne de set
        set_num = 0
        for y in sorted(lines.keys()):
            line_words = sorted(lines[y], key=lambda w: w['x0'])
            scores = [w['text'] for w in line_words if w['text'].isdigit() and 10 <= int(w['text']) <= 50]
            
            if len(scores) >= 2:
                set_num += 1
                if set_num <= 5:
                    sets.append({
                        'numero': set_num,
                        'score_a': int(scores[0]),
                        'score_b': int(scores[1]) if len(scores) > 1 else 0
                    })
        
        return sets
    
    def _parse_arbitres(self, words: list) -> list:
        """Parse les arbitres."""
        arbitres = []
        
        text_full = ' '.join(w['text'] for w in words)
        
        # Pattern: "1er: NOM" ou "2ème: NOM" ou "Marqueur: NOM"
        patterns = [
            (r'1er:\s*([A-ZÀÂÄÉÈÊËÏÎÔÙÛÜÇ][a-zàâäéèêëïîôùûüç]+(?:\s+[A-ZÀÂÄÉÈÊËÏÎÔÙÛÜÇ][a-zàâäéèêëïîôùûüç]+)*)', 'PREMIER'),
            (r'2ème:\s*([A-ZÀÂÄÉÈÊËÏÎÔÙÛÜÇ][a-zàâäéèêëïîôùûüç]+(?:\s+[A-ZÀÂÄÉÈÊËÏÎÔÙÛÜÇ][a-zàâäéèêëïîôùûüç]+)*)', 'SECOND'),
            (r'Marqueur:\s*([A-ZÀÂÄÉÈÊËÏÎÔÙÛÜÇ][a-zàâäéèêëïîôùûüç]+(?:\s+[A-ZÀÂÄÉÈÊËÏÎÔÙÛÜÇ][a-zàâäéèêëïîôùûüç]+)*)', 'MARQUEUR'),
        ]
        
        for pattern, role in patterns:
            match = re.search(pattern, text_full)
            if match:
                arbitres.append({'nom': match.group(1), 'role': role})
        
        return arbitres
    
    def _parse_joueurs(self, words: list, tables: list) -> dict:
        """Parse les joueurs des deux équipes."""
        joueurs = {'equipe_a': [], 'equipe_b': []}
        
        # Les joueurs sont dans les tables avec format: N°, NOM, Prénom, Licence
        # Zone typique: y entre 100 et 400
        
        mid_x = 400  # Séparation approximative gauche/droite
        
        for table in tables:
            for row in table:
                if not row or len(row) < 3:
                    continue
                
                # Chercher pattern: numéro, nom, licence
                first = str(row[0]).strip() if row[0] else ''
                if first.isdigit() and 1 <= int(first) <= 20:
                    joueur = {
                        'numero': int(first),
                        'nom': row[1] if len(row) > 1 else None,
                        'prenom': row[2] if len(row) > 2 else None,
                        'licence': row[3] if len(row) > 3 else None
                    }
                    joueurs['equipe_a'].append(joueur)
        
        return joueurs

# Instancier le parser
parser_v3 = MatchSheetParserV3()
print(f"Parser créé: {parser_v3.name} v{parser_v3.version}")

Parser créé: MatchSheetParserV3 v3.0.0


In [15]:
# Tester le parseur V3 sur un PDF
result_v3 = parser_v3.parse(pdf_path)

print(f"=== Résultat Parser V3 ===")
print(f"Succès: {result_v3['success']}")
print(f"Temps: {result_v3['parse_time_ms']:.2f} ms")
print(f"Erreurs: {result_v3['errors']}")
print(f"\n--- Header ---")
pprint(result_v3['data'].get('header', {}))
print(f"\n--- Équipes ---")
pprint(result_v3['data'].get('equipes', {}))
print(f"\n--- Résultat ---")
pprint(result_v3['data'].get('resultat', {}))
print(f"\n--- Sets ---")
pprint(result_v3['data'].get('sets', []))
print(f"\n--- Arbitres ---")
pprint(result_v3['data'].get('arbitres', []))

=== Résultat Parser V3 ===
Succès: True
Temps: 1612.93 ms
Erreurs: []

--- Header ---
{'code_match': None,
 'competition': 'CHAMPIONNAT',
 'date': None,
 'heure': None,
 'journee': None,
 'lieu': 'Mardi à h Salle: SENIOR | MASCULIN Ligue ILE-DE-FRANCE xxxxx xxxxx S '
         'Début: Fin: S Début: Fin: Ordre de Service E I II III IV V VI I II '
         'III IV V VI E I II III IV V VI I II III IV V VI Formation de Départ',
 'ligue': 'Ligue ILE-DE-FRANCE xxxxx xxxxx S Début: Fin: S Début: Fin: Ordre '
          'de Service E I II III IV V VI I II III IV V VI E I II III IV V VI I '
          'II III IV V VI Formation de Départ',
 'saison': None,
 'salle': 'SENIOR | MASCULIN Ligue ILE-DE-FRANCE xxxxx xxxxx S Début: Fin: S '
          'Début: Fin: Ordre de Service E I II III IV V VI I II III IV V VI E '
          'I II III IV V VI I II III IV V VI Formation de Départ'}

--- Équipes ---
{'equipe_a': None, 'equipe_b': None}

--- Résultat ---
{'duree_totale': None, 'score_final': None, 'vainq

## 3. Comparaison V2 (PyMuPDF) vs V3 (pdfplumber)

In [16]:
# Charger le parser V2
import sys
sys.path.insert(0, '../src')

from pyvolley.parsers.v2 import MatchSheetParserV2

parser_v2 = MatchSheetParserV2()
print(f"Parser V2: {parser_v2.name} v{parser_v2.version}")

Parser V2: MatchSheetParserV2 v2.1.0


In [17]:
# Comparer sur tous les PDFs
comparison_results = []

for pdf_file in PDF_FILES:
    print(f"\n{'='*60}")
    print(f"Fichier: {pdf_file.name}")
    print('='*60)
    
    # Parser V2
    result_v2 = parser_v2.parse(pdf_file)
    
    # Parser V3
    result_v3 = parser_v3.parse(pdf_file)
    
    # Comparer
    comparison = {
        'file': pdf_file.name,
        'v2_success': result_v2.success,
        'v3_success': result_v3['success'],
        'v2_time_ms': result_v2.parse_time_ms,
        'v3_time_ms': result_v3['parse_time_ms'],
        'v2_completeness': result_v2.completeness,
        'v2_match': result_v2.match,
        'v3_data': result_v3['data']
    }
    comparison_results.append(comparison)
    
    # Afficher comparaison
    print(f"\n--- Performance ---")
    print(f"V2: {result_v2.parse_time_ms:.2f}ms | V3: {result_v3['parse_time_ms']:.2f}ms")
    
    print(f"\n--- Extraction ---")
    if result_v2.match:
        m = result_v2.match
        print(f"V2 - Code: {m.code_match}, Équipes: {m.equipe_a.nom if m.equipe_a else '?'} vs {m.equipe_b.nom if m.equipe_b else '?'}")
        print(f"     Score: {m.score_final}, Vainqueur: {m.vainqueur_nom}")
        print(f"     Sets: {[(s.score_a, s.score_b) for s in m.sets]}")
    
    v3_data = result_v3['data']
    print(f"V3 - Code: {v3_data.get('header', {}).get('code_match')}, Équipes: {v3_data.get('equipes', {}).get('equipe_a')} vs {v3_data.get('equipes', {}).get('equipe_b')}")
    print(f"     Score: {v3_data.get('resultat', {}).get('score_final')}, Vainqueur: {v3_data.get('resultat', {}).get('vainqueur')}")
    print(f"     Sets: {[(s.get('score_a'), s.get('score_b')) for s in v3_data.get('sets', [])]}")


Fichier: LIIDF_PMAB001.pdf

--- Performance ---
V2: 32.71ms | V3: 1471.09ms

--- Extraction ---
V3 - Code: None, Équipes: None vs None
     Score: None, Vainqueur: None
     Sets: []

Fichier: LIIDF_PFAA002.pdf

--- Performance ---
V2: 32.71ms | V3: 1471.09ms

--- Extraction ---
V3 - Code: None, Équipes: None vs None
     Score: None, Vainqueur: None
     Sets: []

Fichier: LIIDF_PFAA002.pdf

--- Performance ---
V2: 33.50ms | V3: 1911.24ms

--- Extraction ---
V3 - Code: None, Équipes: None vs None
     Score: 3/0, Vainqueur: SPORTING
     Sets: [(13, 25), (19, 25), (16, 25), (10, 48)]

Fichier: LIIDF_PMAA005.pdf

--- Performance ---
V2: 33.50ms | V3: 1911.24ms

--- Extraction ---
V3 - Code: None, Équipes: None vs None
     Score: 3/0, Vainqueur: SPORTING
     Sets: [(13, 25), (19, 25), (16, 25), (10, 48)]

Fichier: LIIDF_PMAA005.pdf

--- Performance ---
V2: 33.30ms | V3: 1903.27ms

--- Extraction ---
V3 - Code: None, Équipes: None vs None
     Score: 3/0, Vainqueur: PLESSIS-ROBINSON
 

In [18]:
# Résumé de la comparaison
import pandas as pd

summary = []
for c in comparison_results:
    summary.append({
        'Fichier': c['file'],
        'V2 OK': '✓' if c['v2_success'] else '✗',
        'V3 OK': '✓' if c['v3_success'] else '✗',
        'V2 (ms)': f"{c['v2_time_ms']:.1f}",
        'V3 (ms)': f"{c['v3_time_ms']:.1f}",
        'V2 Complétude': f"{c['v2_completeness']*100:.0f}%",
    })

df = pd.DataFrame(summary)
print("\n=== RÉSUMÉ COMPARATIF ===")
print(df.to_string(index=False))

# Moyennes
avg_v2_time = sum(c['v2_time_ms'] for c in comparison_results) / len(comparison_results)
avg_v3_time = sum(c['v3_time_ms'] for c in comparison_results) / len(comparison_results)
v2_success_rate = sum(1 for c in comparison_results if c['v2_success']) / len(comparison_results)
v3_success_rate = sum(1 for c in comparison_results if c['v3_success']) / len(comparison_results)

print(f"\n=== STATISTIQUES ===")
print(f"V2 - Temps moyen: {avg_v2_time:.2f}ms, Taux succès: {v2_success_rate*100:.0f}%")
print(f"V3 - Temps moyen: {avg_v3_time:.2f}ms, Taux succès: {v3_success_rate*100:.0f}%")
print(f"\nRapport vitesse V2/V3: {avg_v2_time/avg_v3_time:.2f}x" if avg_v3_time > 0 else "")


=== RÉSUMÉ COMPARATIF ===
          Fichier V2 OK V3 OK V2 (ms) V3 (ms) V2 Complétude
LIIDF_PMAB001.pdf     ✗     ✓    32.7  1471.1            0%
LIIDF_PFAA002.pdf     ✗     ✓    33.5  1911.2            0%
LIIDF_PMAA005.pdf     ✗     ✓    33.3  1903.3            0%
LIIDF_PMAA003.pdf     ✗     ✓    30.8  1817.4            0%
LIIDF_PFAA001.pdf     ✗     ✓    30.5  1887.2            0%
LIIDF_PMAA001.pdf     ✗     ✓    34.9  2085.0            0%
LIIDF_PMAA004.pdf     ✗     ✓    31.7  1716.5            0%
LIIDF_PMAA002.pdf     ✗     ✓    35.3  2152.7            0%
LIIDF_1MAA001.pdf     ✗     ✓    41.9  2130.1            0%

=== STATISTIQUES ===
V2 - Temps moyen: 33.85ms, Taux succès: 0%
V3 - Temps moyen: 1897.16ms, Taux succès: 100%

Rapport vitesse V2/V3: 0.02x


## 4. Import dans la Base de Données

In [19]:
# Utiliser le parser V2 (plus mature) pour l'import
from pyvolley.database.connection import DatabaseSession
from pyvolley.database.import_service import MatchImportService

# Parser tous les matchs avec V2
matches_to_import = []

for pdf_file in PDF_FILES:
    result = parser_v2.parse(pdf_file)
    if result.success and result.match:
        matches_to_import.append(result.match)
        print(f"✓ {pdf_file.name}: {result.match.code_match}")
    else:
        print(f"✗ {pdf_file.name}: {result.errors}")

print(f"\n{len(matches_to_import)} matchs prêts à importer")

✗ LIIDF_PMAB001.pdf: ["Erreur de parsing: 1 validation error for Equipe\nnom\n  String should have at least 2 characters [type=string_too_short, input_value='', input_type=str]\n    For further information visit https://errors.pydantic.dev/2.10/v/string_too_short"]
✗ LIIDF_PFAA002.pdf: ["Erreur de parsing: 1 validation error for Equipe\nnom\n  String should have at least 2 characters [type=string_too_short, input_value='S', input_type=str]\n    For further information visit https://errors.pydantic.dev/2.10/v/string_too_short"]
✗ LIIDF_PMAA005.pdf: ["Erreur de parsing: 1 validation error for Equipe\nnom\n  String should have at least 2 characters [type=string_too_short, input_value='S', input_type=str]\n    For further information visit https://errors.pydantic.dev/2.10/v/string_too_short"]
✗ LIIDF_PMAA003.pdf: ["Erreur de parsing: 1 validation error for Equipe\nnom\n  String should have at least 2 characters [type=string_too_short, input_value='S', input_type=str]\n    For further infor

In [20]:
# Importer dans la base de données
with DatabaseSession() as session:
    service = MatchImportService(session)
    
    imported = 0
    errors = []
    
    for match in matches_to_import:
        try:
            match_db = service.import_match(match)
            imported += 1
            print(f"✓ Importé: {match.code_match} (ID: {match_db.id})")
        except Exception as e:
            errors.append((match.code_match, str(e)))
            print(f"✗ Erreur {match.code_match}: {e}")
    
    session.commit()
    print(f"\n=== RÉSULTAT ===")
    print(f"Importés: {imported}/{len(matches_to_import)}")
    if errors:
        print(f"Erreurs: {len(errors)}")


=== RÉSULTAT ===
Importés: 0/0


In [21]:
# Vérifier le contenu de la base
from pyvolley.database.models import MatchDB, SetDB, EquipeDB, SaisonDB, CompetitionDB
from sqlalchemy import select

with DatabaseSession() as session:
    matchs = session.scalars(select(MatchDB)).all()
    
    print(f"=== {len(matchs)} matchs en base ===")
    for m in matchs:
        print(f"\n{m.code_match} - {m.date_match}")
        print(f"  {m.equipe_a.nom} vs {m.equipe_b.nom}")
        print(f"  Score: {m.score_final} | Vainqueur: {m.vainqueur_nom}")
        print(f"  Sets: {[(s.score_a, s.score_b) for s in m.sets]}")

2026-02-01 22:34:57,800 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-02-01 22:34:57,921 INFO sqlalchemy.engine.Engine SELECT matchs.id, matchs.code_match, matchs.date_match, matchs.heure_match, matchs.lieu, matchs.salle, matchs.journee, matchs.competition_id, matchs.saison_id, matchs.equipe_a_id, matchs.equipe_b_id, matchs.vainqueur_id, matchs.vainqueur_nom, matchs.score_final, matchs.sets_equipe_a, matchs.sets_equipe_b, matchs.duree_totale, matchs.remarques, matchs.source_pdf, matchs.parsed_at, matchs.created_at, matchs.updated_at 
FROM matchs
2026-02-01 22:34:57,923 INFO sqlalchemy.engine.Engine [generated in 0.00332s] ()
2026-02-01 22:34:57,921 INFO sqlalchemy.engine.Engine SELECT matchs.id, matchs.code_match, matchs.date_match, matchs.heure_match, matchs.lieu, matchs.salle, matchs.journee, matchs.competition_id, matchs.saison_id, matchs.equipe_a_id, matchs.equipe_b_id, matchs.vainqueur_id, matchs.vainqueur_nom, matchs.score_final, matchs.sets_equipe_a, matchs.sets_equipe_b, 